# 04 · Map the QUBO to an Ising Hamiltonian

## Question

Why can a binary optimization problem be written as an Ising Hamiltonian?

## Concept

**Physics / quantum view.** A Hamiltonian is an energy operator that generates quantum evolution. An Ising model assigns energies to configurations of interacting two-valued spins. Here those values are encoded as the eigenvalues of the single-qubit Pauli $Z$ operator. The cost Hamiltonian uses these energies to represent the optimization objective.

On a computational basis state, the Pauli $Z$ operator has eigenvalue $+1$ for $\lvert0\rangle$ and $-1$ for $\lvert1\rangle$. Thus $q_i=(1-Z_i)/2$ gives bit 0 or 1, respectively. Write the QUBO as $c+\sum_i l_iq_i+\sum_{i<j}w_{ij}q_iq_j$. Substitution gives

$$
l_iq_i=\frac{l_i}{2}-\frac{l_i}{2}Z_i,\qquad
w_{ij}q_iq_j=\frac{w_{ij}}{4}(1-Z_i-Z_j+Z_iZ_j).
$$

The first term contributes a field and a constant. The second contributes a coupling, two fields, and a constant. Collect these into $E_\mathrm{QUBO}=c_\mathrm{Ising}+H_C$, where $H_C$ contains only $Z$ and $ZZ$ terms.

## Check / exercise

With $A=5$, what is the coefficient of $Z_0Z_1$? Does adding the same constant to every energy affect the ground state?

## Worked example

In [1]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

J = np.array([[-1.0, -3.0], [-2.5, -1.0]])  # Illustrative performance
A = 5.0
constant = 2 * A
linear = -J.ravel() - A
quadratic = {(0, 1): 2 * A, (2, 3): 2 * A}

offset = constant + linear.sum() / 2
fields = -linear / 2
couplings = {}
for (i, j), weight in quadratic.items():
    offset += weight / 4
    fields[i] -= weight / 4
    fields[j] -= weight / 4
    couplings[i, j] = weight / 4

H_C = SparsePauliOp.from_sparse_list(
    [("Z", [i], h) for i, h in enumerate(fields)]
    + [("ZZ", [i, j], coupling) for (i, j), coupling in couplings.items()],
    num_qubits=4,
)
print("Fields:", fields)
print("Couplings:", couplings)
print("Constant offset:", offset)

Fields: [-0.5  -1.5  -1.25 -0.5 ]
Couplings: {(0, 1): 2.5, (2, 3): 2.5}
Constant offset: 8.75


## Result

Qiskit writes the highest-index qubit on the left of a basis label. Reverse that label to recover the teaching order $q_{11},q_{12},q_{21},q_{22}$ before evaluating the original objective.

In [2]:
ising = H_C.to_matrix().diagonal().real
qubo = []
print("Teaching order | QUBO | Ising without offset | Difference")
for index, energy in enumerate(ising):
    label = f"{index:04b}"[::-1]
    bits = np.array([int(bit) for bit in label]).reshape(2, 2)
    direct = float(-(J * bits).sum() + A * ((bits.sum(axis=1) - 1)**2).sum())
    qubo.append(direct)
    print(f"{label:14} | {direct:5.2f} | {energy:20.2f} | {direct - energy:10.2f}")

np.testing.assert_allclose(qubo, ising + offset, atol=1e-12)
assert np.array_equal(np.sign(np.subtract.outer(qubo, qubo)),
                      np.sign(np.subtract.outer(ising, ising)))
assert f"{int(np.argmin(ising)):04b}"[::-1] == "1001"
print("All 16 energies and their ordering agree.")

Teaching order | QUBO | Ising without offset | Difference
0000           | 10.00 |                 1.25 |       8.75
1000           |  6.00 |                -2.75 |       8.75
0100           |  8.00 |                -0.75 |       8.75
1100           | 14.00 |                 5.25 |       8.75
0010           |  7.50 |                -1.25 |       8.75
1010           |  3.50 |                -5.25 |       8.75
0110           |  5.50 |                -3.25 |       8.75
1110           | 11.50 |                 2.75 |       8.75
0001           |  6.00 |                -2.75 |       8.75
1001           |  2.00 |                -6.75 |       8.75
0101           |  4.00 |                -4.75 |       8.75
1101           | 10.00 |                 1.25 |       8.75
0011           | 13.50 |                 4.75 |       8.75
1011           |  9.50 |                 0.75 |       8.75
0111           | 11.50 |                 2.75 |       8.75
1111           | 17.50 |                 8.75 |       8.7

**Exercise check.** The coupling is $2A/4=2.5$. The offset cancels in every energy difference, so all rankings and ties are preserved.

## Takeaway

The QUBO and Ising formulations assign the same relative energies to all 16 states.

## Next step

[Notebook 05](05_qaoa_2x2.ipynb) uses the verified cost Hamiltonian to build a circuit.

### Presentation asset

Use the linear and quadratic substitutions alongside the energy comparison table. Highlight the constant difference shared by all rows.

MkDocs topic for this notebook: `docs/ising.md`.